In [14]:
import numpy as np
import tensorflow as tf
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax.training import train_state
import time

In [2]:
from Preprocess import prepare_tcn_data
from TCN_model import TCN, init_train_state, train_step

In [3]:
with tf.io.gfile.GFile("gs://fi2010-lob-data/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt", 'r') as f:
    raw_data = np.loadtxt(f)

In [4]:
batch_size = 1024
time_step = 127

In [5]:
train_dataset = prepare_tcn_data(raw_data, batch_size, time_step, True, False)

E0000 00:00:1775632554.470987   90535 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
len(train_dataset)

38

In [12]:
rng = jax.random.PRNGKey(3721)
rng, dropout_rng = jax.random.split(rng)
model = TCN()
state, dropout_rng = init_train_state(rng, model)

In [16]:
EPOCHS = 1
start_time = time.time()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    steps = 0
    
    for batch_x, batch_y in train_dataset.as_numpy_iterator():
        
        x_jax = jnp.asarray(batch_x, dtype=jnp.bfloat16)
        y_jax = jnp.asarray(batch_y, dtype=jnp.int32)
        
        state, loss, accuracy = train_step(state, x_jax, y_jax, dropout_rng)
        
        epoch_loss += loss.item()
        steps += 1
        
        # 测试模式小贴士：每跑几步打印一下，确保模型没卡死，Loss 没变成 NaN
        if steps % 10 == 0:
            print(f"  ⚡ Step {steps} | 实时 Loss: {loss.item():.4f} | Acc: {accuracy.item():.4f}")

    avg_loss = epoch_loss / max(1, steps)
    print(f'time:{time.time() - start_time}')
    print(f"✅ Epoch {epoch+1}/{EPOCHS} 完美收官！")
    print(f"📊 总计运行步数: {steps} | 平均 Loss: {avg_loss:.4f}")
    print("-" * 50)

  ⚡ Step 10 | 实时 Loss: 1.1616 | Acc: 0.4453
  ⚡ Step 20 | 实时 Loss: 1.2003 | Acc: 0.3818
  ⚡ Step 30 | 实时 Loss: 1.1504 | Acc: 0.4326
time:203.63557052612305
✅ Epoch 1/1 完美收官！
📊 总计运行步数: 38 | 平均 Loss: 1.1924
--------------------------------------------------
